<a href="https://colab.research.google.com/github/vaddadisravani201/DubaiLandCoverClassification/blob/main/DubaiLandCoverClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dubai Land Cover Classification using Sentinel-2 and Machine Learning

This notebook demonstrates a machine learning workflow for land cover classification of Dubai, UAE using satellite imagery from Sentinel-2 processed through Google Earth Engine and visualized with geemap.

The workflow begins by generating a cloud-masked Sentinel-2 composite for the period January–June 2025. Several spectral indices are then extracted to improve feature representation, including NDVI (vegetation), NDWI (water), NDBI (urban/built-up), and NDSI (desert/bare land). These indices are combined with the original spectral bands to create a feature stack.

Automatic labeling is performed using threshold-based rules derived from the spectral indices. Stratified sampling is applied to generate a balanced training dataset for machine learning.

Two models are implemented:

Artificial Neural Network (ANN) built with TensorFlow

Random Forest classifier using the Google Earth Engine machine learning library

Model performance is evaluated using a confusion matrix and classification report from scikit-learn. The final classified land cover map identifies four major classes:

Urban Infrastructure Area

Desert / Bare Land

Water Bodies

Vegetated Area

The results are visualized interactively using geemap, allowing comparison between the Sentinel-2 RGB imagery and the machine-learning-based land cover classification.

In [ ]:
# 1. INSTALL LIBRARIES
!pip install earthengine-api geemap tensorflow scikit-learn -q

# 2. IMPORT LIBRARIES
import ee
import geemap
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# 3. AUTHENTICATE & INITIALIZE EE
ee.Authenticate()
ee.Initialize(project='satelliteanalysis')
print("Earth Engine initialized successfully.")

# 4. LOAD DUBAI BOUNDARY
adm1 = ee.FeatureCollection("FAO/GAUL/2015/level1")
dubai = adm1.filter(ee.Filter.And(
    ee.Filter.eq('ADM0_NAME','United Arab Emirates'),
    ee.Filter.eq('ADM1_NAME','Dubai')
))
dubai_geom = dubai.geometry()
print("Dubai boundary loaded.")

# 5. CLOUD MASK FUNCTION
def maskS2clouds(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
           qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask)

# 6. LOAD SENTINEL-2 COMPOSITE (Jan-Jun 2025)
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(dubai_geom)
      .filterDate('2025-01-01', '2025-06-30')
      .map(maskS2clouds)
      .median()
      .clip(dubai_geom))
image = s2.select(['B2','B3','B4','B8','B11','B12'])
print("Sentinel-2 composite ready.")

# 7. CALCULATE INDICES
# NDVI for vegetation
ndvi = image.normalizedDifference(['B8','B4']).rename('NDVI')

# NDBI for built-up (urban)
ndbi = image.normalizedDifference(['B11','B8']).rename('NDBI')

# NDWI for water
ndwi = image.normalizedDifference(['B3','B8']).rename('NDWI')

# Normalized Difference Sand Index (NDSI) for desert
ndsi = image.normalizedDifference(['B11','B4']).rename('NDSI')

# Stack indices with original bands
image_indices = image.addBands([ndvi, ndwi, ndbi, ndsi])

# 8. AUTOMATIC CLASS LABELING BASED ON INDICES (thresholds tuned for Dubai)
def assign_class(img):
    return img.expression(
        "NDVI >= 0.2 ? 3" +          # Vegetation
        ": NDWI >= 0.05 ? 2" +      # Water
        ": NDBI >= 0.1 ? 1" +       # Urban
        ": NDSI >= 0.05 ? 0" +      # Desert
        ": 0",
        {
            'NDVI': img.select('NDVI'),
            'NDWI': img.select('NDWI'),
            'NDBI': img.select('NDBI'),
            'NDSI': img.select('NDSI')
        }
    ).rename('class')

labeled_image = assign_class(image_indices)

# 9. SAMPLE THE IMAGE STRATIFIED (balanced)
training_samples = image_indices.addBands(labeled_image).stratifiedSample(
    numPoints=2000,
    classBand='class',
    classValues=[0,1,2,3],
    classPoints=[500,500,500,500],
    region=dubai_geom,
    scale=10,
    geometries=False
)

# 10. CONVERT TO PANDAS
training_dict = training_samples.getInfo()
rows = [f['properties'] for f in training_dict['features']]
df = pd.DataFrame(rows)
print("Class Distribution:")
print(df['class'].value_counts())

# 11. PREPARE DATA FOR ANN
feature_columns = ['B2','B3','B4','B8','B11','B12','NDVI','NDWI','NDBI','NDSI']
X = df[feature_columns].values / 10000.0  # Normalize reflectance
y = df['class'].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 12. BUILD & TRAIN OPTIMIZED ANN
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(len(feature_columns),)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(4, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
callback = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
model.fit(X_train, y_train, epochs=100, validation_split=0.2, callbacks=[callback])

# 13. EVALUATE ANN
y_pred = np.argmax(model.predict(X_test), axis=1)
print("ANN Test Accuracy:", model.evaluate(X_test, y_test, verbose=0)[1])
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

# 14. RANDOM FOREST CLASSIFICATION FOR DUBAI USING INDICES
classifier = ee.Classifier.smileRandomForest(200).train(
    features=training_samples,
    classProperty='class',
    inputProperties=feature_columns
)
classified_dubai = image_indices.classify(classifier).clip(dubai_geom)

# 15. VISUALIZE WITH GEEMAP
Map = geemap.Map(center=[25.276987, 55.296249], zoom=10)
rgb_vis = {'bands':['B4','B3','B2'], 'min':0, 'max':3000}
class_vis = {'min':0,'max':3,'palette':['yellow','red','blue','green']}

Map.addLayer(image.clip(dubai_geom), rgb_vis, "Sentinel-2 RGB")
Map.add_basemap('SATELLITE')
Map.addLayer(classified_dubai, class_vis, "Land Cover Classification")


Map.add_legend(
    title="Land Cover Classes",
    colors=[(255,255,0),(255,0,0),(0,0,255),(0,255,0)],
    labels=["Urban Infrastructure Area","Desert/Bare Land","Water Bodies","Vegetated Area"]
)
Map.addLayerControl()
Map

Earth Engine initialized successfully.
Dubai boundary loaded.
Sentinel-2 composite ready.
Class Distribution:
class
0    500
1    500
2    500
3    500
Name: count, dtype: int64
Epoch 1/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.3048 - loss: 1.3705 - val_accuracy: 0.2500 - val_loss: 1.3181
Epoch 2/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4045 - loss: 1.2630 - val_accuracy: 0.7094 - val_loss: 1.0740
Epoch 3/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7021 - loss: 0.9931 - val_accuracy: 0.8531 - val_loss: 0.7034
Epoch 4/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7371 - loss: 0.7201 - val_accuracy: 0.9250 - val_loss: 0.4309
Epoch 5/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7782 - loss: 0.5285 - val_accuracy: 0.9094 - val_loss: 0.3788
Epoch 6/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8317 - loss: 0.4710 - val_accuracy: 0.9156 - val_loss: 0.3272
Epoch 7/100
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accurac

Map(center=[25.276987, 55.296249], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Sear…